# RAG Hybrid Search with Pinecone

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key_pc = os.getenv("PINECONE_API_KEY")

In [23]:
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search"
pc = Pinecone(api_key=api_key_pc)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # dimension for dense vector (all-MiniLM-L6-v2 = 384)
        metric='dotproduct',  # sparse values only support dotproduct
        spec=ServerlessSpec(cloud='aws', region="us-east-1")
    )

In [24]:
index = pc.Index(index_name)
index

Index(host='https://hybrid-search-y3apscs.svc.aped-4627-b74a.pinecone.io')

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [26]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()

In [27]:
sentences = [
    "in 2023, I visited Paris",
    "in 2022, I visited New York",
    "in 2021, I visited WDC",
    "in 2024, I visited London",
    "in 2025, I visited Ohio"
]

## Fit BM25 & Generate Vectors

In [28]:
# Fit BM25 on the corpus (learns term frequencies)
bm25_encoder.fit(sentences)

# Generate dense vectors (semantic meaning)
dense_vectors = embeddings.embed_documents(sentences)

# Generate sparse vectors (keyword matching)
sparse_vectors = bm25_encoder.encode_documents(sentences)

print(f"Dense vector dim: {len(dense_vectors[0])}")
print(f"Sparse vector example: {sparse_vectors[0]}")

  0%|          | 0/5 [00:00<?, ?it/s]

Dense vector dim: 384
Sparse vector example: {'indices': [3871233692, 2395889254, 4239951674], 'values': [0.4664723032069971, 0.4664723032069971, 0.4664723032069971]}


## Upsert to Pinecone 

In [29]:
# Build the upsert payload
vectors = []
for i, (dense, sparse) in enumerate(zip(dense_vectors, sparse_vectors)):
    vectors.append({
        "id": str(i),
        "values": dense,
        "sparse_values": sparse,
        "metadata": {"text": sentences[i]}
    })

# Upsert using keyword argument (Pinecone SDK v5+ requires this)
index.upsert(vectors=vectors)
print(f"Upserted {len(vectors)} vectors")

Upserted 5 vectors


In [35]:
# Verify the index 
index

Index(host='https://hybrid-search-y3apscs.svc.aped-4627-b74a.pinecone.io')

## Hybrid Search Query
Query using both dense and sparse vectors with an **alpha** parameter to control the weighting:
- `alpha = 1.0` → pure semantic (dense only)
- `alpha = 0.0` → pure keyword (sparse only)
- `alpha = 0.5` → balanced hybrid

In [31]:
def hybrid_query(query_text, top_k=5, alpha=0.5):
    """
    Perform hybrid search combining dense and sparse vectors.
    
    Args:
        query_text: The search query string
        top_k: Number of results to return
        alpha: Weight for dense vs sparse (1.0 = pure dense, 0.0 = pure sparse)
    """
    # Generate dense query vector
    dense_query = embeddings.embed_query(query_text)
    
    # Generate sparse query vector
    sparse_query = bm25_encoder.encode_queries(query_text)
    
    # Scale vectors by alpha for weighted hybrid search
    scaled_dense = [v * alpha for v in dense_query]
    scaled_sparse = {
        "indices": sparse_query["indices"],
        "values": [v * (1 - alpha) for v in sparse_query["values"]]
    }
    
    # Query Pinecone with both dense and sparse vectors
    results = index.query(
        vector=scaled_dense,
        sparse_vector=scaled_sparse,
        top_k=top_k,
        include_metadata=True
    )
    
    return results

In [32]:
# Example: Balanced hybrid search (alpha=0.5)
query = "which city did I visit in 2024?"
results = hybrid_query(query, alpha=0.5)

print(f"Query: {query}\n")
for match in results["matches"]:
    print(f"  Score: {match['score']:.4f} | {match['metadata']['text']}")

Query: which city did I visit in 2024?

  Score: 0.2996 | in 2025, I visited Ohio
  Score: 0.2349 | in 2021, I visited WDC
  Score: 0.3496 | in 2022, I visited New York
  Score: 0.5075 | in 2024, I visited London
  Score: 0.3443 | in 2023, I visited Paris


In [33]:
# Example: Pure semantic search (alpha=1.0)
results_dense = hybrid_query("places I traveled to", alpha=1.0)

print("Pure Semantic Search:\n")
for match in results_dense["matches"]:
    print(f"  Score: {match['score']:.4f} | {match['metadata']['text']}")

Pure Semantic Search:

  Score: 0.3966 | in 2025, I visited Ohio
  Score: 0.3567 | in 2021, I visited WDC
  Score: 0.4759 | in 2022, I visited New York
  Score: 0.4843 | in 2024, I visited London
  Score: 0.4518 | in 2023, I visited Paris


In [34]:
# Example: Pure keyword search (alpha=0.0)
results_sparse = hybrid_query("Paris", alpha=0.0)

print("Pure Keyword Search:\n")
for match in results_sparse["matches"]:
    print(f"  Score: {match['score']:.4f} | {match['metadata']['text']}")

Pure Keyword Search:

  Score: 0.0000 | in 2025, I visited Ohio
  Score: 0.0000 | in 2021, I visited WDC
  Score: 0.0000 | in 2022, I visited New York
  Score: 0.0000 | in 2024, I visited London
  Score: 0.4665 | in 2023, I visited Paris
